<a href="https://colab.research.google.com/github/sanAkel/ufs_diurnal_diagnostics/blob/main/RTOFS/binary_nc_converter/read_archive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install cartopy

In [ ]:
import os
import requests
import tarfile

import xarray as xr
import numpy as np

from datetime import datetime, timedelta

import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def download_file(url, local_filename):
    """Downloads a file in chunks to handle large binary data safely."""
    print(f"Downloading: {url} ...", end=" ", flush=True)
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status() # Check for 404 or other errors
            with open(local_filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print("Done.")
    except Exception as e:
        print(f"\nFailed to download {url}: {e}")

In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

## Following datasets are needed:
- Grid
- Topography

They can be downloaded from [HYCOM datasets.](https://data.hycom.org/datasets/GLBb0.08/expt_93.0/topo/)

## Download HYCOM output data from [RTOFS.](https://noaa-nws-rtofs-pds.s3.amazonaws.com/index.html#rtofs.20260226/)

## Configuration

In [ ]:
# Path to "fix" files - if they are not there, they will be downloaded.
hycom_datasets_path = '/content/drive/MyDrive/datasets/RTOFS/v2.5/'

# Path to data files that will be downloaded
data_path="/content/drive/MyDrive/datasets/tmp/RTOFS/v2.5/"

data_date = "20260226" # date

# what files to download/plot
#file_types = ["rtofs_glo.t00z.n00.archs", "rtofs_glo.t00z.n00.archv"]
file_types = ["rtofs_glo.t00z.n-24.archs", "rtofs_glo.t00z.n-24.archv"]

## Download grid, topography (if not already present)

In [ ]:
hycom_datasets_url_base = "https://data.hycom.org/datasets/GLBb0.08/expt_93.0/topo/"
file_bases = ['regional.grid', 'depth_GLBb0.08_09m11']
extensions = ['.a', '.b']
output_dir = hycom_datasets_path

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Iterate through combinations and check for existence
for base in file_bases:
    for ext in extensions:
        filename = f"{base}{ext}"
        local_path = os.path.join(output_dir, filename)

        if os.path.exists(local_path):
            print(f"File already exists: {local_path}")
        else:
            url = f"{hycom_datasets_url_base}{filename}"
            download_file(url, local_path)

print("\nHYCOM dataset check/download complete.")

In [ ]:
base_url = f"https://noaa-nws-rtofs-pds.s3.amazonaws.com/rtofs.{data_date}/"
suffixes = [".a.tgz", ".b"]

# Create a sub-folder for the specific date
dest_path = os.path.join(data_path, data_date)
os.makedirs(dest_path, exist_ok=True)

for f_type in file_types:
    for suffix in suffixes:
        filename = f"{f_type}{suffix}"
        url = f"{base_url}{filename}"
        local_path = os.path.join(dest_path, filename)

        # Download the file
        download_file(url, local_path)

        # Uncompress if it is a .tgz file
        if suffix == ".a.tgz" and os.path.exists(local_path):
            print(f"Extracting {filename} ...", end=" ")
            try:
                with tarfile.open(local_path, "r:gz") as tar:
                    tar.extractall(path=dest_path)
                print("Done.")
                # Remove the .tgz file after successful extraction
                os.remove(local_path)
                print(f"Removed {filename}.")
            except Exception as e:
                print(f"\nFailed to extract or remove {filename}: {e}")

### These functions are based on those at [pyhycom.](https://github.com/uwincm/pyhycom/blob/master/pyhycom.py)

In [ ]:
# return file handle
def open_a_file(filename, mode):
    file = open(filename[:-1]+'a',mode=mode)
    return file

#Return the name of the corresponding HYCOM "b" file.
def get_b_filename(fName):
    bfilename = fName[:-1]+'b'
    return bfilename

#Return a list where each element contains text from each line of `b file`
def getTextFile(fName):
    return [line.rstrip() for line in open(fName,'r').readlines()]

In [ ]:
# get dimensions of an archive from .b file
def getDims(fName, topo_file=False):

  f = getTextFile(get_b_filename(fName))
  idmFound, jdmFound = [False, False]

  if topo_file:
    for line in f:
        if 'i/jdm' in line:
          xx = line.split()[3]; jdm = xx[0:4]
          idm = line.split()[2]
          idmFound = True
          jdmFound = True
        if idmFound and jdmFound:break
  else:
    for line in f:
        if 'idm' in line:
          idm = line.split()[0]
          idmFound = True
        if 'jdm' in line:
          jdm = line.split()[0]
          jdmFound = True
        if idmFound and jdmFound:break

  return int(jdm), int(idm)

In [ ]:
def getFieldIndex(field, fName):
    f = getTextFile(get_b_filename(fName))
    if 'arch' in fName.split('/')[-1]:f = f[10:] # skip first 10 lines
    if 'grid' in fName.split('/')[-1]:f = f[3:] # skip first 3 lines
    fieldIndex = []
    for line in f:
      if field == line.split()[0].replace('.','').replace(':',''):
        fieldIndex.append(f.index(line))
    return fieldIndex

In [ ]:
def getField(field, fName, undef=np.nan, x_range=None, y_range=None):

  dims = getDims(fName)
  if dims.__len__() == 2:
    jdm, idm = dims
  else:
    jdm, idm, kdm = dims
    print("\n-- CAUTION! Read 3d archive is not yet ready!\n")

  reclen = 4*idm*jdm # Record length in bytes
  # HYCOM binary data is written out in chunks/"words" of multiples of 4096*4 bytes.
  # Length of one level of one variable (reclen) will be between
  # consecutive multiples of the wordlen. Data is padded to bring the volume
  # up to the next multiple. The "pad" value below is equal to the bytes that are needed to do this.
  wordlen = 4096*4
  pad = wordlen * np.ceil(reclen / wordlen) - reclen   # Pad size in bytes
  fieldRecords = getFieldIndex(field,fName)         # Get field record indices
  fieldAddresses = np.array(fieldRecords)*(reclen+pad) # Address in bytes

  file = open_a_file(fName,mode='rb') # Open file
  if dims.__len__() == 2: # 2-d field
    field = np.zeros((jdm,idm))
    file.seek(int(fieldAddresses[0]),0) # Move to address of the field
    data = file.read(idm*jdm*4)
    field = np.reshape(np.frombuffer(data, dtype='float32', count=idm*jdm),(jdm,idm)).byteswap()

    if not x_range is None:
      field = field[:,:,x_range]
    if not y_range is None:
      field = field[:,y_range,:]

  #field = field.byteswap() # Convert to little-endian
  file.close()
  field[field == np.float32(2**100)] = undef

  return field

In [ ]:
# Number of records in the binary file, read from .b
def getNumberOfRecords(fName):
  f = getTextFile(get_b_filename(fName))
  if 'arch' in fName:
      f = f[10:]; return len(f)
  if 'grid' in fName:
      f = f[3:]; return len(f)
  if 'depth' in fName:
      return 1
  if 'restart' in fName:
      f = f[2:]; return len(f)

In [ ]:
def getBathymetry(grid_fName, topog_fName, undef=np.nan):

  jdm,idm = getDims(grid_fName)

  file = open_a_file(topog_fName, mode='rb')
  #Data is in float32, which has 4 bytes/value
  data = file.read(idm*jdm*4)
  field = np.reshape(np.frombuffer(data,dtype='float32',count=idm*jdm).byteswap(),(jdm,idm))
  file.close()

  print(f"field.shape={field.shape}")
  field[field>2**99] = undef

  return field

In [ ]:
bFile=get_b_filename(hycom_datasets_path+"/regional.grid.a")
print(f"Reading {bFile}\n")

jm, im = getDims(hycom_datasets_path+"/regional.grid.a")
print(f"im={im}, jm={jm}")

jm, im = getDims(hycom_datasets_path+"/depth_GLBb0.08_09m11.a", topo_file=True)
print(f"im={im}, jm={jm}")

In [ ]:
fn1 = hycom_datasets_path+"/regional.grid.a"
print(getFieldIndex('plat',fn1))
print("\n")

#fn2 = data_path+"/v2.4_rtofs_glo.t00z.n00.archs.a"
#print(getFieldIndex('srfhgt',fn2))
#print(getFieldIndex('salin',fn2))

In [ ]:
print(getNumberOfRecords(hycom_datasets_path+"/regional.grid.a"))
print("\n")
print(getNumberOfRecords(hycom_datasets_path+"/depth_GLBb0.08_09m11.a"))
print("\n")
print(getNumberOfRecords(dest_path+"/rtofs_glo.t00z.n-24.archs.a"))

In [ ]:
depth=getBathymetry(hycom_datasets_path+"/regional.grid.a", hycom_datasets_path+"/depth_GLBb0.08_09m11.a")

In [ ]:
plt.imshow(depth, origin='lower')
plt.colorbar()

In [ ]:
plon = getField('plon', hycom_datasets_path+"/regional.grid.a")
plat = getField('plat', hycom_datasets_path+"/regional.grid.a")

# variables
sst = getField('temp',   dest_path+"/rtofs_glo.t00z.n-24.archs.a")
sss = getField('salin',  dest_path+"/rtofs_glo.t00z.n-24.archs.a")
ssh = getField('srfhgt', dest_path+"/rtofs_glo.t00z.n-24.archs.a")

uvel = getField('u-vel', dest_path+"/rtofs_glo.t00z.n-24.archs.a")
vvel = getField('v-vel', dest_path+"/rtofs_glo.t00z.n-24.archs.a")

In [ ]:
# SSH in m
grav = 9.81
ssh = ssh/grav #https://github.com/HYCOM/HYCOM-src/blob/5c9f48918374965e2df09042180ea8a69c650dc4/mod_cb_arrays.F90#L90

In [ ]:
plt.figure(figsize=(10,10))

plt.subplot(2,2,1)
plt.imshow(sst, origin='lower'); plt.colorbar(shrink=0.4)

plt.subplot(2,2,2)
plt.imshow(sss, origin='lower'); plt.colorbar(shrink=0.4)

plt.subplot(2,2,3)
plt.imshow(ssh, origin='lower'); plt.colorbar(shrink=0.4)

plt.subplot(2,2,4)
plt.imshow( np.sqrt(uvel**2 + vvel**2), origin='lower'); plt.colorbar(shrink=0.4)

In [ ]:
!cat $dest_path/rtofs_glo.t00z.n-24.archs.b

## Create (xarray) dataset to simply life!

In [ ]:
def hycom_date_to_datetime(hycom_model_day):
    """Converts HYCOM model day (days since 1901-01-01) to a datetime object."""
    # HYCOM model days start from 1901-01-01.
    start_date = datetime(1901, 1, 1)

    # Extract integer part for days and fractional part for hours
    full_days = int(hycom_model_day)
    fractional_day = hycom_model_day - full_days

    # Add the full number of days directly to the start_date
    target_datetime = start_date + timedelta(days=full_days)
    target_datetime += timedelta(hours=fractional_day * 24)

    return target_datetime

In [ ]:
def get_model_day(fName, varName):
  bFile = getTextFile( get_b_filename(fName))
  for line in bFile:
    if varName in line:
      model_day = line.split()[3]
  return model_day

In [ ]:
model_day_value = float(get_model_day(dest_path+"/rtofs_glo.t00z.n00.archs.a", 'srfhgt'))

actual_datetime = hycom_date_to_datetime(model_day_value)

# Format the datetime object to match the expected string format
formatted_datetime_str = actual_datetime.strftime('%Y/%m/%d %H:%M:%S')

print(formatted_datetime_str)

In [ ]:
ds = xr.Dataset()

# Get the actual datetime from the model day
model_day_value = float(get_model_day(dest_path+"/rtofs_glo.t00z.n00.archs.a", 'srfhgt'))
actual_datetime = hycom_date_to_datetime(model_day_value)

ds['Latitude'] = xr.DataArray(plat,
    dims=("Y", "X"),
    coords={"Y": np.arange(1, plat.shape[0]+1), "X": np.arange(1, plat.shape[1]+1)},
    name="Latitude",
    attrs={"units": "degrees_north", "long_name": "latitude"})

ds['Longitude'] = xr.DataArray(plon,
    dims=("Y", "X"),
    coords={"Y": np.arange(1, plon.shape[0]+1), "X": np.arange(1, plon.shape[1]+1)},
    name="Longitude",
    attrs={"units": "degrees_east", "long_name": "longitude", "modulo": "360 degrees"})

ds['time'] = np.array([actual_datetime], dtype='datetime64')

ds['srfhgt'] = xr.DataArray(ssh,
    dims=("Y", "X"),
    coords={"Y": np.arange(1, ssh.shape[0]+1), "X": np.arange(1, ssh.shape[1]+1)},
    name="srfhgt",
    attrs={"units": "m", "long_name": "surface height"})

ds.attrs = {"source": "NCEP RTOFS v2.5",
            "history": "converted archive to netcdf",
            "using": "https://github.com/NOAA-EMC/RTOFS_GLO"}

ds=ds.set_coords(["Latitude", "Longitude", "time"])

In [ ]:
ds

## Plot few regions/projections:
- N Pole (Arctic).
- Globe.
- S Pole (Antarctic).

In [ ]:
fig = plt.figure(figsize=[8,6])

ax = fig.add_subplot(1,1,1, projection=ccrs.NorthPolarStereo(central_longitude=-30.0))
ax.add_feature(cfeature.LAND, facecolor='grey', alpha=0.1)
ax.coastlines(color='k', alpha=0.1)
ax.set_extent([-300, 60, 50, 90], ccrs.PlateCarree())

im = ds.srfhgt.plot(ax=ax, x='Longitude', y='Latitude',
            vmin=-1, vmax=1, cmap='bwr',
            transform=ccrs.PlateCarree(),
            add_labels=False, add_colorbar=False)

im.axes.gridlines(color='black', alpha=0.5, linestyle='--', draw_labels=True)

cax = ax.inset_axes([0.8, 0.3, 0.03, 0.6])
fig.colorbar(im, cax=cax, orientation='vertical', ticks=[-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1.])
cax.tick_params(labelsize=10, rotation=0)
cax.set_title('{} [{}]'.format(ds.srfhgt.attrs['long_name'], ds.srfhgt.attrs['units']))
#cax.set_title('{}/{}/{}'.format(ds.time.dt.year.values[0], ds.time.dt.month.values[0], ds.time.dt.day.values[0]))

In [ ]:
fig = plt.figure(figsize=[8,6])

ax = fig.add_subplot(1,1,1, projection=ccrs.Miller(central_longitude=-180.0))
ax.add_feature(cfeature.LAND, facecolor='grey', alpha=0.1)
ax.coastlines(color='k', alpha=0.1)
#ax.set_extent([-300, 60, -60, 60], ccrs.PlateCarree())

im = ds.srfhgt.plot(ax=ax, x='Longitude', y='Latitude',
            vmin=-1, vmax=1, cmap='bwr',
            transform=ccrs.PlateCarree(),
            add_labels=False, add_colorbar=False)

im.axes.gridlines(color='black', alpha=0.5, linestyle='--', draw_labels=True)

cax = ax.inset_axes([0.02, 0.05, 0.4, 0.03])
fig.colorbar(im, cax=cax, orientation='horizontal', ticks=[-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1.])
cax.tick_params(labelsize=8, rotation=0)
cax.set_title('{} [{}]'.format(ds.srfhgt.attrs['long_name'], ds.srfhgt.attrs['units']))
#cax.set_title('{}/{}/{}'.format(ds.time.dt.year.values[0], ds.time.dt.month.values[0], ds.time.dt.day.values[0]))

In [ ]:
fig = plt.figure(figsize=[8,6])

ax = fig.add_subplot(1,1,1, projection=ccrs.SouthPolarStereo(central_longitude=-120.0))
ax.add_feature(cfeature.LAND, facecolor='grey', alpha=0.1)
ax.coastlines(color='k', alpha=0.1)
ax.set_extent([-300, 60, -40, -90], ccrs.PlateCarree())

im = ds.srfhgt.plot(ax=ax, x='Longitude', y='Latitude',
                  vmin=-1, vmax=1, cmap='bwr',
                  transform=ccrs.PlateCarree(),
                 add_labels=False, add_colorbar=False)

im.axes.gridlines(color='black', alpha=0.5, linestyle='--', draw_labels=True)

cax = ax.inset_axes([0.3, 0.45, 0.35, 0.03])
fig.colorbar(im, cax=cax, orientation='horizontal', ticks=[-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1.])
cax.tick_params(labelsize=6, rotation=30)
cax.set_title('{} [{}]'.format(ds.srfhgt.attrs['long_name'], ds.srfhgt.attrs['units']))
##cax.set_title('{}/{}/{}'.format(ds.time.dt.year.values[0], ds.time.dt.month.values[0], ds.time.dt.day.values[0]))

In [ ]:
fig = plt.figure(figsize=[18, 12])
gs = gridspec.GridSpec(2, 2, figure=fig, width_ratios=[1.2, 1])

# --- Column 1: Globe (Miller) spanning both rows ---
ax1 = fig.add_subplot(gs[:, 0], projection=ccrs.Miller(central_longitude=-180.0))
ax1.add_feature(cfeature.LAND, facecolor='grey', alpha=0.1)
ax1.coastlines(color='k', alpha=0.1)
im1 = ds.srfhgt.plot(ax=ax1, x='Longitude', y='Latitude',
                    vmin=-1, vmax=1, cmap='bwr',
                    transform=ccrs.PlateCarree(),
                    add_labels=False, add_colorbar=False)
ax1.gridlines(color='black', alpha=0.3, linestyle='--', draw_labels=True)
cax1 = ax1.inset_axes([0.02, 0.05, 0.4, 0.02])
fig.colorbar(im1, cax=cax1, orientation='horizontal', ticks=[-1, -0.5, 0, 0.5, 1])
cax1.set_title(f"{ds.srfhgt.attrs['long_name']} [{ds.srfhgt.attrs['units']}]", fontsize=10)
#ax1.set_title("Global (Miller)")

# --- Column 2, Row 1: North Polar Stereo ---
ax2 = fig.add_subplot(gs[0, 1], projection=ccrs.NorthPolarStereo(central_longitude=-30.0))
ax2.add_feature(cfeature.LAND, facecolor='grey', alpha=0.1)
ax2.coastlines(color='k', alpha=0.1)
ax2.set_extent([-300, 60, 50, 90], ccrs.PlateCarree())
im2 = ds.srfhgt.plot(ax=ax2, x='Longitude', y='Latitude',
                    vmin=-1, vmax=1, cmap='bwr',
                    transform=ccrs.PlateCarree(),
                    add_labels=False, add_colorbar=False)
ax2.gridlines(color='black', alpha=0.3, linestyle='--', draw_labels=True)
cax2 = ax2.inset_axes([0.85, 0.3, 0.03, 0.6])
fig.colorbar(im2, cax=cax2, orientation='vertical', ticks=[-1, -0.5, 0, 0.5, 1])
cax2.set_title('SSH [m]', fontsize=8)
#ax2.set_title("North Pole")

# --- Column 2, Row 2: South Polar Stereo ---
ax3 = fig.add_subplot(gs[1, 1], projection=ccrs.SouthPolarStereo(central_longitude=-120.0))
ax3.add_feature(cfeature.LAND, facecolor='grey', alpha=0.1)
ax3.coastlines(color='k', alpha=0.1)
ax3.set_extent([-300, 60, -40, -90], ccrs.PlateCarree())
im3 = ds.srfhgt.plot(ax=ax3, x='Longitude', y='Latitude',
                    vmin=-1, vmax=1, cmap='bwr',
                    transform=ccrs.PlateCarree(),
                    add_labels=False, add_colorbar=False)
ax3.gridlines(color='black', alpha=0.3, linestyle='--', draw_labels=True)
cax3 = ax3.inset_axes([0.3, 0.15, 0.4, 0.03])
fig.colorbar(im3, cax=cax3, orientation='horizontal', ticks=[-1, -0.5, 0, 0.5, 1])
cax3.set_title('SSH [m]', fontsize=8)
#ax3.set_title("South Pole")

plt.tight_layout()
plt.show()